## SCRIPT TO OBTAIN THE PRECIPITATION-RELATED FIGURES OF THE PAPER

In [1]:
import xarray as xr
import numpy as np
import cartopy
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append('/home/pfernand/Postdoc/Python_scripts/')
from WAM_functions import *
from Forced_simulation_functions import *
from scipy import stats
import numpy.ma as ma
import matplotlib.mlab as mlab
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import BoundaryNorm
import scipy
import matplotlib.patches as patches
import metpy.calc as mpcalc
from metpy.units import units
import cftime
from regr_sig import *
from regr_2d_ttest import *
import xskillscore
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel
from matplotlib.ticker import MultipleLocator

In [2]:
###### PARÁMETROS IMPORTANTES #######

sahel_box = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':10}
WA_box = {'lat_min':-10, 'lat_max':40, 'lon_min':-30, 'lon_max':50}

sahel_box_west = {'lat_min':12, 'lat_max':17, 'lon_min':-17, 'lon_max':-7}
sahel_box_east = {'lat_min':8, 'lat_max':17, 'lon_min':-7, 'lon_max':10}
sahel_box_north = {'lat_min':12.5, 'lat_max':17, 'lon_min':-17, 'lon_max':10}
sahel_box_south = {'lat_min':8, 'lat_max':12.5, 'lon_min':-17, 'lon_max':10}
sahel_box = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':10}

time_limits = ['1950-01-01T12:00:00.000000000', '2014-12-31T12:00:00.000000000']

#### PERÍODOS DE ESTUDIO ##############

N_forced = 180 # Número de días en MJJASO en 360d
N_coupled = 185 # Número de días en MJJASO en DAMIP

period_1 = [1950, 1970]
period_2 = [1970, 1990]
period_3 = [1994, 2014]

#######################################
font = {'family' : 'calibri',
       'weight': 'bold',
       'size': 50}
plt.rc('font', **font)
plt.rc('axes',edgecolor='k',linewidth=3)

In [8]:
# Some useful functions ###

def n_I_rainy_days(ds, variable, threshold):

    """ Función que calcula el número de días lluviosos y su intensidad por año así como la cantidad de precipitación caída durante los días no lluviosos """


    ds['masked_rainy'] = xr.where(ds.pr > threshold, 1, 0)
    ds['masked_nonrainy'] = xr.where(ds.pr <= threshold, 1, 0)
    ds['pr_rainy'] = xr.where(ds.pr > threshold, ds.pr, 0)
    ds['pr_rainy_cum_yr'] = ds.pr_rainy.groupby('time.year').sum(dim = 'time', skipna = True)
    ds['n_rainy'] = ds.masked_rainy.groupby('time.year').sum(dim = 'time', skipna = True)
    ds['I_rainy_yr'] = ds.pr_rainy_cum_yr / ds.n_rainy
    ds['pr_nonrainy_cum_yr'] = xr.where(ds.pr <= threshold, ds.pr, 0).groupby('time.year').sum(dim = 'time', skipna = True)
    ds['pr_cum_yr'] = ds.pr.groupby('time.year').sum(dim = 'time', skipna = True)

    return ds



def WAM_moderate_heavy_extreme_difquant_forced(ds, variable, threshold):

    """ Función que calcula el número de días con eventos moderados, fuertes y extremos con percentiles calculados con todos los miembros de una simulación a lo largo de todos los tiempos, variable es la precipitación en mm/day. 
    Para la simulación acoplada utilizamos todos los tiempos.
    Para las simulationes forzadas, el umbral se calcula con la simulación de control y se aplica a las variantes DA y OM."""

    only_wet_days = xr.where((ds[variable] > threshold), ds[variable], np.nan)
    

    # También saco el año de incio

    if 'member' in list(ds.dims):
        perc_95 = only_wet_days.quantile(0.95, dim = ('time','member','simulation'), skipna = True)
        perc_75 = only_wet_days.quantile(0.75, dim = ('time','member', 'simulation'), skipna = True)
        
        ds['n_extreme'] = ((only_wet_days > perc_95.values)).groupby('time.year').sum(dim = 'time', skipna = True)
        ds['n_heavy'] = ((only_wet_days >= perc_75.values)).groupby('time.year').sum(dim = 'time', skipna = True)
        ds['n_moderate'] = ((only_wet_days < perc_75.values)).groupby('time.year').sum(dim = 'time', skipna = True)

        return ds


    if 'simulation' not in list(ds.dims):
        perc_95 = only_wet_days.quantile(0.95, dim = ('time'), skipna = True)
        perc_75 = only_wet_days.quantile(0.75, dim = ('time'), skipna = True)
        
        ds['n_extreme'] = ((only_wet_days > perc_95.values)).groupby('time.year').sum(dim = 'time', skipna = True)
        ds['n_heavy'] = ((only_wet_days >= perc_75.values)).groupby('time.year').sum(dim = 'time', skipna = True)
        ds['n_moderate'] = ((only_wet_days < perc_75.values)).groupby('time.year').sum(dim = 'time', skipna = True)

        return ds


    else:     
        perc_95 = only_wet_days.isel(simulation = [0,1,2]).quantile(0.95, dim = ('time', 'simulation'), skipna = True)
        perc_75 = only_wet_days.isel(simulation = [0,1,2]).quantile(0.75, dim = ('time', 'simulation'), skipna = True)
        
        ds['n_extreme'] = ((only_wet_days > perc_95.values)).groupby('time.year').sum(dim = 'time', skipna = True)
        ds['n_heavy'] = ((only_wet_days >= perc_75.values)).groupby('time.year').sum(dim = 'time', skipna = True)
        ds['n_moderate'] = ((only_wet_days < perc_75.values)).groupby('time.year').sum(dim = 'time', skipna = True)


        return ds


## Plot precipitation changes in the historical simulation

In [ ]:
ds_hist = xr.open_dataset('/data/pfernand/Postdoc/IPSL/IPSL-CM6A-LR/WA_box/hist-EXT/Precipitation/pr_day_IPSL-CM6A-LR_historical_all_members_gr_18500101-20291231_WA_mm_day.nc').sel(time = slice(time_limits[0], time_limits[1]))
ds_aer = xr.open_dataset('/data/pfernand/Postdoc/IPSL/IPSL-CM6A-LR/WA_box/hist-aer/pr_day_IPSL-CM6A-LR_hist-aer_all_members_gr_19500101-20141231_WA_mm_day.nc')
ds_ghg = xr.open_dataset('/data/pfernand/Postdoc/IPSL/IPSL-CM6A-LR/WA_box/hist-GHG/pr_day_IPSL-CM6A-LR_hist-ghg_all_members_gr_19500101-20141231_WA_mm_day.nc')

# Selecciono los 3 períodos 

ds_aer1960 = ds_aer.sel(time = slice('1950-01-01T12:00:00.000000000','1970-12-31T12:00:00.000000000'))
ds_aer1980 = ds_aer.sel(time = slice('1970-01-01T12:00:00.000000000','1990-12-31T12:00:00.000000000'))
ds_aer2004 = ds_aer.sel(time = slice('1994-01-01T12:00:00.000000000','2014-12-31T12:00:00.000000000'))


ds_ghg1960 = ds_ghg.sel(time = slice('1950-01-01T12:00:00.000000000','1970-12-31T12:00:00.000000000'))
ds_ghg1980 = ds_ghg.sel(time = slice('1970-01-01T12:00:00.000000000','1990-12-31T12:00:00.000000000'))
ds_ghg2004 = ds_ghg.sel(time = slice('1994-01-01T12:00:00.000000000','2014-12-31T12:00:00.000000000'))


ds_hist1960 = ds_hist.sel(time = slice('1950-01-01T12:00:00.000000000','1970-12-31T12:00:00.000000000'))
ds_hist1980 = ds_hist.sel(time = slice('1970-01-01T12:00:00.000000000','1990-12-31T12:00:00.000000000'))
ds_hist2004 = ds_hist.sel(time = slice('1994-01-01T12:00:00.000000000','2014-12-31T12:00:00.000000000'))


# Y selecciono JAS 

JAS_aer1960 = ds_aer1960.time.dt.month.isin(range(7, 10))
ds_aer1960_JAS = ds_aer1960.isel(time = JAS_aer1960)
ds_aer1960_JAS_yearly = ds_aer1960_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


JAS_aer1980 = ds_aer1980.time.dt.month.isin(range(7, 10))
ds_aer1980_JAS = ds_aer1980.isel(time = JAS_aer1980)
ds_aer1980_JAS_yearly = ds_aer1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


JAS_aer2004 = ds_aer2004.time.dt.month.isin(range(7, 10))
ds_aer2004_JAS = ds_aer2004.isel(time = JAS_aer2004)
ds_aer2004_JAS_yearly = ds_aer2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)



JAS_ghg1960 = ds_ghg1960.time.dt.month.isin(range(7, 10))
ds_ghg1960_JAS = ds_ghg1960.isel(time = JAS_ghg1960)
ds_ghg1960_JAS_yearly = ds_ghg1960_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


JAS_ghg1980 = ds_ghg1980.time.dt.month.isin(range(7, 10))
ds_ghg1980_JAS = ds_ghg1980.isel(time = JAS_ghg1980)
ds_ghg1980_JAS_yearly = ds_ghg1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


JAS_ghg2004 = ds_ghg2004.time.dt.month.isin(range(7, 10))
ds_ghg2004_JAS = ds_ghg2004.isel(time = JAS_ghg2004)
ds_ghg2004_JAS_yearly = ds_ghg2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)



JAS_hist1960 = ds_hist1960.time.dt.month.isin(range(7, 10))
ds_hist1960_JAS = ds_hist1960.isel(time = JAS_hist1960)
ds_hist1960_JAS_yearly = ds_hist1960_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


JAS_hist1980 = ds_hist1980.time.dt.month.isin(range(7, 10))
ds_hist1980_JAS = ds_hist1980.isel(time = JAS_hist1980)
ds_hist1980_JAS_yearly = ds_hist1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


JAS_hist2004 = ds_hist2004.time.dt.month.isin(range(7, 10))
ds_hist2004_JAS = ds_hist2004.isel(time = JAS_aer2004)
ds_hist2004_JAS_yearly = ds_hist2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


In [ ]:
# Pongo máscara 

sea_mask = sea_mask_creator(ds_aer1960_JAS)

ds_aer1960_JAS_yearly = ds_aer1960_JAS_yearly.where(sea_mask == 0)
ds_aer1980_JAS_yearly = ds_aer1980_JAS_yearly.where(sea_mask == 0)
ds_aer2004_JAS_yearly = ds_aer2004_JAS_yearly.where(sea_mask == 0)


ds_ghg1960_JAS_yearly = ds_ghg1960_JAS_yearly.where(sea_mask == 0)
ds_ghg1980_JAS_yearly = ds_ghg1980_JAS_yearly.where(sea_mask == 0)
ds_ghg2004_JAS_yearly = ds_ghg2004_JAS_yearly.where(sea_mask == 0)



ds_hist1960_JAS_yearly = ds_hist1960_JAS_yearly.where(sea_mask == 0)
ds_hist1980_JAS_yearly = ds_hist1980_JAS_yearly.where(sea_mask == 0)
ds_hist2004_JAS_yearly = ds_hist2004_JAS_yearly.where(sea_mask == 0)

ds_hist1980_JAS = ds_hist1980_JAS.where(sea_mask == 0)

ds_hist2004_JAS = ds_hist2004_JAS.where(sea_mask == 0)


In [ ]:
# Calculo la significación estadística de las diferencias de la precipitación total 

t_stat_aer_pr_19601980, p_value_aer_pr_19601980 = stats.ttest_ind(ds_aer1960_JAS_yearly.values, ds_aer1980_JAS_yearly.values, axis = 0, equal_var=False)
t_stat_aer_pr_19802004, p_value_aer_pr_19802004 = stats.ttest_ind(ds_aer1980_JAS_yearly.values, ds_aer2004_JAS_yearly.values, axis = 0, equal_var=False)

t_stat_ghg_pr_19601980, p_value_ghg_pr_19601980 = stats.ttest_ind(ds_ghg1960_JAS_yearly.values, ds_ghg1980_JAS_yearly.values, axis = 0, equal_var=False)
t_stat_ghg_pr_19802004, p_value_ghg_pr_19802004 = stats.ttest_ind(ds_ghg1980_JAS_yearly.values, ds_ghg2004_JAS_yearly.values, axis = 0, equal_var=False)


t_stat_hist_pr_19601980, p_value_hist_pr_19601980 = stats.ttest_ind(ds_hist1960_JAS_yearly.values, ds_hist1980_JAS_yearly.values, axis = 0, equal_var=False)
t_stat_hist_pr_19802004, p_value_hist_pr_19802004 = stats.ttest_ind(ds_hist1980_JAS_yearly.values, ds_hist2004_JAS_yearly.values, axis = 0, equal_var=False)


t_stat_hist_n_19802004, p_value_hist_n_19802004 = stats.ttest_ind(ds_hist1980_JAS.n_rainy.mean(dim = 'year', skipna = True), ds_hist2004_JAS.n_rainy.mean(dim = 'year', skipna = True), axis = 0, equal_var=False)
t_stat_hist_I_19802004, p_value_hist_I_19802004 = stats.ttest_ind(ds_hist1980_JAS.I_rainy_yr.mean(dim = 'year', skipna = True), ds_hist2004_JAS.I_rainy_yr.mean(dim = 'year', skipna = True), axis = 0, equal_var=False)


In [ ]:
ds_hist1980_JAS  = n_I_rainy_days(ds_hist1980_JAS, 'pr',1)
ds_hist2004_JAS  = n_I_rainy_days(ds_hist2004_JAS, 'pr',1)

ds_hist1980_JAS['I_rainy'] = ds_hist1980_JAS.pr_rainy_cum_yr.mean(dim = ('year','member'), skipna = True) / ds_hist1980_JAS.n_rainy.mean(dim = ('year', 'member'), skipna = True)
ds_hist2004_JAS['I_rainy'] = ds_hist2004_JAS.pr_rainy_cum_yr.mean(dim = ('year', 'member'), skipna = True) / ds_hist2004_JAS.n_rainy.mean(dim = ('year', 'member'), skipna = True)


In [ ]:
ds_hist1980_JAS = ds_hist1980_JAS.where(sea_mask == 0)
ds_hist2004_JAS = ds_hist2004_JAS.where(sea_mask == 0)


In [ ]:
ds_hist2004_JAS_yearly.mean(dim = ('year'), skipna = True).shape

In [ ]:

vmin_value = -1
vmax_value = 1

sahel_box = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':10}
sahel_box_west = {'lat_min':12.5, 'lat_max':17, 'lon_min':-17, 'lon_max':-10}
sahel_box_east = {'lat_min':8, 'lat_max':17, 'lon_min':5, 'lon_max':10}
extent = [-20,30,0,25]

levs = np.arange(-0.7, 0.8, 0.1)
levs_n = np.arange(-5,6,1)
levs_I = np.arange(-1.2,1.4,0.2)
levels_p_clim = np.asarray([1,2,4,6,8,10,12])
levels_n_clim = np.asarray([0,15,35,55,75,90])
levels_I_clim = np.asarray([0,4,6,8, 10,16])

fig = plt.figure(figsize=(40,20))


ft = 60

ax1 = fig.add_subplot(1,1,1, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_hist2004_JAS_yearly.lon.values, ds_hist2004_JAS_yearly.lat.values, ds_hist2004_JAS_yearly.mean(dim = ('year'), skipna = True) - ds_hist1980_JAS_yearly.mean(dim = ('year'), skipna = True), levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())

cbar = plt.colorbar(cax, ax = ax1, orientation='horizontal', fraction=0.08,pad=0.08)
cbar.set_label(r'mm·d$^{-1}$',fontsize = ft)
cbar.ax.tick_params(labelsize=ft)


CS = ax1.contour(ds_hist2004_JAS_yearly.lon.values, ds_hist2004_JAS_yearly.lat.values, (ds_hist2004_JAS_yearly.mean(dim = ('year'), skipna = True).values + ds_hist1980_JAS_yearly.mean(dim = ('year'), skipna = True).values)/ 2, levels = levels_p_clim, colors = 'black', linewidths = 3)

ax1.clabel(
    CS,
    inline=True,
    fontsize=50,
)



ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box_east['lon_min'], sahel_box_east['lon_max'], sahel_box_east['lon_max'], sahel_box_east['lon_min'], sahel_box_east['lon_min']],  [sahel_box_east['lat_max'], sahel_box_east['lat_max'], sahel_box_east['lat_min'], sahel_box_east['lat_min'], sahel_box_east['lat_max']], linewidth = 7, color = 'purple')
ax1.plot([sahel_box_west['lon_min'], sahel_box_west['lon_max'], sahel_box_west['lon_max'], -15, sahel_box_west['lon_min'], sahel_box_west['lon_min']],  [sahel_box_west['lat_max'], sahel_box_west['lat_max'], sahel_box_west['lat_min'], sahel_box_west['lat_min'], 12.5, sahel_box_west['lat_max']], linewidth = 7, color = 'orange')


LON, LAT = np.meshgrid(ds_hist1960_JAS.lon.values, ds_hist1960_JAS.lat.values)
lon_masked = ma.masked_where((np.isnan(p_value_hist_pr_19802004)) | (p_value_hist_pr_19802004 > 0.05), LON)
lat_masked = ma.masked_where((np.isnan(p_value_hist_pr_19802004)) | (p_value_hist_pr_19802004 > 0.05), LAT)
hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)


ax1.set_title(r'[1994-2014] $-$ [1970-1990] Precipitation change', weight = 'bold', fontsize = ft+10) 
ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=50)
plt.rc('ytick', labelsize=50)


plt.savefig('/home/pfernand/Postdoc/Results/Forced_simulations/JAS_precipitation_changes_hist.jpg', bbox_inches = 'tight', dpi = 150)



## Compute JAS precipitation changes and contributions from number of rainy days and intensity

In [ ]:

####### AA-RELATED SIMULATIONS ###########


ds_ctrlaer1980 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/CTRLAER1980ann/DA/CTRLAER1980ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_ctrlaer1980 = ds_ctrlaer1980.rename({'precip': 'pr'})
ds_ctrlaer1980 = ds_ctrlaer1980.sel(lat=ds_ctrlaer1980.lat[::-1])

ds_ctrlaer2004 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/CTRLAER2004ann/DA/CTRLAER2004ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_ctrlaer2004 = ds_ctrlaer2004.rename({'precip': 'pr'})
ds_ctrlaer2004 = ds_ctrlaer2004.sel(lat=ds_ctrlaer2004.lat[::-1])


ds_aerda2004 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/AERDA2004ann/DA/AERDA2004ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_aerda2004 = ds_aerda2004.rename({'precip': 'pr'})
ds_aerda2004 = ds_aerda2004.sel(lat=ds_aerda2004.lat[::-1])


ds_aerom2004 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/AEROM2004ann/DA/AEROM2004ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_aerom2004 = ds_aerom2004.rename({'precip': 'pr'})
ds_aerom2004 = ds_aerom2004.sel(lat=ds_aerom2004.lat[::-1])

# También cargo el fichero DAMIP de hist-aer
ds_aer = xr.open_dataset('/data/pfernand/Postdoc/IPSL/IPSL-CM6A-LR/WA_box/hist-aer/pr_day_IPSL-CM6A-LR_hist-aer_all_members_gr_19500101-20141231_WA_mm_day.nc').sel(lat = slice(WA_box['lat_min'], WA_box['lat_max']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).sel(time = slice(time_limits[0], time_limits[1]))
ds_aer1980 = ds_aer.sel(time = slice('1970-01-01T12:00:00.000000000','1990-12-31T12:00:00.000000000'))
ds_aer2004 = ds_aer.sel(time = slice('1994-01-01T12:00:00.000000000','2014-12-31T12:00:00.000000000'))


######## GHG-RELATED SIMULATIONS ##############

ds_ctrlghg1980 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/CTRLGHG1980ann/DA/CTRLGHG1980ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_ctrlghg1980 = ds_ctrlghg1980.rename({'precip': 'pr'})
ds_ctrlghg1980 = ds_ctrlghg1980.sel(lat=ds_ctrlghg1980.lat[::-1])

ds_ctrlghg2004 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/CTRLGHG2004ann/DA/CTRLGHG2004ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_ctrlghg2004 = ds_ctrlghg2004.rename({'precip': 'pr'})
ds_ctrlghg2004 = ds_ctrlghg2004.sel(lat=ds_ctrlghg2004.lat[::-1])


ds_ghgda2004 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/GHGDA2004ann/DA/GHGDA2004ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_ghgda2004 = ds_ghgda2004.rename({'precip': 'pr'})
ds_ghgda2004 = ds_ghgda2004.sel(lat=ds_ghgda2004.lat[::-1])


ds_ghgom2004 = xr.open_dataset('/thredds/tgcc/work/fernandb/Hard_links/GHGOM2004ann/DA/GHGOM2004ann_19800101_20791230_1D_histday.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})
ds_ghgom2004 = ds_ghgom2004.rename({'precip': 'pr'})
ds_ghgom2004 = ds_ghgom2004.sel(lat=ds_ghgom2004.lat[::-1])

# También cargo el fichero DAMIP de hist-aer
ds_ghg = xr.open_dataset('/data/pfernand/Postdoc/IPSL/IPSL-CM6A-LR/WA_box/hist-GHG/pr_day_IPSL-CM6A-LR_hist-ghg_all_members_gr_19500101-20141231_WA_mm_day.nc').sel(lat = slice(WA_box['lat_min'], WA_box['lat_max']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).sel(time = slice(time_limits[0], time_limits[1]))
ds_ghg1980 = ds_ghg.sel(time = slice('1970-01-01T12:00:00.000000000','1990-12-31T12:00:00.000000000'))
ds_ghg2004 = ds_ghg.sel(time = slice('1994-01-01T12:00:00.000000000','2014-12-31T12:00:00.000000000'))



########## TOTAL CHANGE SIMULATIONS ########################

ds_ctrl1980 = xr.open_dataset('/thredds/tgcc/store/fernandb/CTRL1980mb/DA/CTRL1980mb_19800101_20791230_1D_histday_pr_mm_day.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})[['precip']]
ds_ctrl1980 = ds_ctrl1980.rename({'precip': 'pr'})
ds_ctrl1980 = ds_ctrl1980.sel(lat=ds_ctrl1980.lat[::-1])

ds_ctrl2004 = xr.open_dataset('/thredds/tgcc/store/fernandb/CTRL2004mb/DA/CTRL2004mb_19800101_20791230_1D_histday_pr_mm_day.nc').sel(lat = slice(WA_box['lat_max'], WA_box['lat_min']), lon = slice(WA_box['lon_min'], WA_box['lon_max'])).rename({'time_counter': 'time'})[['precip']]
ds_ctrl2004 = ds_ctrl2004.rename({'precip': 'pr'})
ds_ctrl2004 = ds_ctrl2004.sel(lat=ds_ctrl1980.lat[::-1])






In [4]:
# Selecting JAS

JAS_ctrlaer1980 = ds_ctrlaer1980.time.dt.month.isin(range(7, 10))
ds_ctrlaer1980_JAS = ds_ctrlaer1980.isel(time = JAS_ctrlaer1980)
ds_ctrlaer1980_JAS['time'] = ds_ctrlaer1980_JAS.time.astype("datetime64[ns]") 
ds_ctrlaer1980_JAS_yearly = ds_ctrlaer1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 

JAS_ctrlaer2004 = ds_ctrlaer2004.time.dt.month.isin(range(7, 10))
ds_ctrlaer2004_JAS = ds_ctrlaer2004.isel(time = JAS_ctrlaer2004)
ds_ctrlaer2004_JAS['time'] = ds_ctrlaer2004_JAS.time.astype("datetime64[ns]") 
ds_ctrlaer2004_JAS_yearly = ds_ctrlaer2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 


JAS_aerda2004 = ds_aerda2004.time.dt.month.isin(range(7, 10))
ds_aerda2004_JAS = ds_aerda2004.isel(time = JAS_aerda2004)
ds_aerda2004_JAS['time'] = ds_aerda2004_JAS.time.astype("datetime64[ns]")
ds_aerda2004_JAS_yearly = ds_aerda2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 


JAS_aerom2004 = ds_aerom2004.time.dt.month.isin(range(7, 10))
ds_aerom2004_JAS = ds_aerom2004.isel(time = JAS_aerom2004)
ds_aerom2004_JAS['time'] = ds_aerom2004_JAS.time.astype("datetime64[ns]")
ds_aerom2004_JAS_yearly = ds_aerom2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 


JAS = ds_aer.time.dt.month.isin(range(7, 10))
ds_aer_JAS = ds_aer.isel(time = JAS)
ds_aer_JAS_yearly = ds_aer_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True)

JAS_aer1980 = ds_aer1980.time.dt.month.isin(range(7, 10))
ds_aer1980_JAS = ds_aer1980.isel(time = JAS_aer1980)
ds_aer1980_JAS_yearly = ds_aer1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)



JAS_aer2004 = ds_aer2004.time.dt.month.isin(range(7, 10))
ds_aer2004_JAS = ds_aer2004.isel(time = JAS_aer2004)
ds_aer2004_JAS['time'] = ds_aer1980_JAS.time
ds_aer2004_JAS_yearly = ds_aer2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)


############# También para los ficheros de GHG  ################

#JAS_ctrlghg1960 = ds_ctrlghg1960.time.dt.month.isin(range(7, 10))
#ds_ctrlghg1960_JAS = ds_ctrlghg1960.isel(time = JAS_ctrlghg1960)
#ds_ctrlghg1960_JAS['time'] = ds_ctrlghg1960_JAS.time.astype("datetime64[ns]")
#ds_ctrlghg1960_JAS_yearly = ds_ctrlghg1960_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 

JAS_ctrlghg1980 = ds_ctrlghg1980.time.dt.month.isin(range(7, 10))
ds_ctrlghg1980_JAS = ds_ctrlghg1980.isel(time = JAS_ctrlghg1980)
ds_ctrlghg1980_JAS['time'] = ds_ctrlghg1980_JAS.time.astype("datetime64[ns]") 
ds_ctrlghg1980_JAS_yearly = ds_ctrlghg1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 

JAS_ctrlghg2004 = ds_ctrlghg2004.time.dt.month.isin(range(7, 10))
ds_ctrlghg2004_JAS = ds_ctrlghg2004.isel(time = JAS_ctrlghg2004)
ds_ctrlghg2004_JAS['time'] = ds_ctrlghg2004_JAS.time.astype("datetime64[ns]") 
ds_ctrlghg2004_JAS_yearly = ds_ctrlghg2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 

#JAS_ghgda1980 = ds_ghgda1980.time.dt.month.isin(range(7, 10))
#ds_ghgda1980_JAS = ds_ghgda1980.isel(time = JAS_ghgda1980)
#ds_ghgda1980_JAS['time'] = ds_ghgda1980_JAS.time.astype("datetime64[ns]") 
#ds_ghgda1980_JAS_yearly = ds_ghgda1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 


JAS_ghgda2004 = ds_ghgda2004.time.dt.month.isin(range(7, 10))
ds_ghgda2004_JAS = ds_ghgda2004.isel(time = JAS_ghgda2004)
ds_ghgda2004_JAS['time'] = ds_ghgda2004_JAS.time.astype("datetime64[ns]")
ds_ghgda2004_JAS_yearly = ds_ghgda2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 


JAS_ghgom2004 = ds_ghgom2004.time.dt.month.isin(range(7, 10))
ds_ghgom2004_JAS = ds_ghgom2004.isel(time = JAS_ghgom2004)
ds_ghgom2004_JAS['time'] = ds_ghgom2004_JAS.time.astype("datetime64[ns]")
ds_ghgom2004_JAS_yearly = ds_ghgom2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 


# And for hist-GHG
JAS = ds_ghg.time.dt.month.isin(range(7, 10))
ds_ghg_JAS = ds_ghg.isel(time = JAS)
ds_ghg_JAS_yearly = ds_ghg_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True)


JAS_ghg1980 = ds_ghg1980.time.dt.month.isin(range(7, 10))
ds_ghg1980_JAS = ds_ghg1980.isel(time = JAS_ghg1980)
ds_ghg1980_JAS_yearly = ds_ghg1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)



JAS_ghg2004 = ds_ghg2004.time.dt.month.isin(range(7, 10))
ds_ghg2004_JAS = ds_ghg2004.isel(time = JAS_ghg2004)
ds_ghg2004_JAS['time'] = ds_ghg1980_JAS.time
ds_ghg2004_JAS_yearly = ds_ghg2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True).mean(dim = 'member', skipna = True)

JAS_ghg = ds_ghg.time.dt.month.isin(range(7, 10))
ds_ghg_JAS = ds_ghg.isel(time = JAS_ghg).mean(dim = 'member', skipna = True)
ds_ghg = ds_ghg.mean(dim = 'member', skipna = True)

JAS_aer = ds_aer.time.dt.month.isin(range(7, 10))
ds_aer_JAS = ds_aer.isel(time = JAS_aer).mean(dim = 'member', skipna = True)
ds_aer = ds_aer.mean(dim = 'member', skipna = True)


JAS_ctrl1980 = ds_ctrl1980.time.dt.month.isin(range(7, 10))
ds_ctrl1980_JAS = ds_ctrl1980.isel(time = JAS_ctrl1980)
ds_ctrl1980_JAS['time'] = ds_ctrl1980_JAS.time.astype("datetime64[ns]") 
ds_ctrl1980_JAS_yearly = ds_ctrl1980_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 

JAS_ctrl2004 = ds_ctrl2004.time.dt.month.isin(range(7, 10))
ds_ctrl2004_JAS = ds_ctrl2004.isel(time = JAS_ctrl2004)
ds_ctrl2004_JAS['time'] = ds_ctrl2004_JAS.time.astype("datetime64[ns]") 
ds_ctrl2004_JAS_yearly = ds_ctrl2004_JAS.pr.groupby('time.year').mean(dim = 'time', skipna = True) 

In [5]:
# MERGING XARRAYS IN ONE

ds_forcedghg_JAS = xr.Dataset({"pr" : xr.concat([ds_ctrlghg1980_JAS.pr, ds_ctrlghg2004_JAS.pr, ds_ghgda2004_JAS.pr, ds_ghgom2004_JAS.pr], dim=xr.DataArray(['CTRLGHG1980', 'CTRLGHG2004', 'GHGDA2004', 'GHGOM2004'], dims='simulation'))})
ds_forcedaer_JAS = xr.Dataset({"pr" : xr.concat([ds_ctrlaer1980_JAS.pr, ds_ctrlaer2004_JAS.pr, ds_aerda2004_JAS.pr, ds_aerom2004_JAS.pr], dim=xr.DataArray(['CTRLAER1980', 'CTRLAER2004', 'AERDA2004', 'AEROM2004'], dims='simulation'))})

ds_forced_JAS = xr.Dataset({"pr" : xr.concat([ds_ctrl1980_JAS.pr, ds_ctrl2004_JAS.pr], dim=xr.DataArray(['CTRL1980', 'CTRL2004'], dims='simulation'))})


ds_coupledghg_JAS = xr.Dataset({"pr" : xr.concat([ds_ghg1980_JAS.pr, ds_ghg2004_JAS.pr], dim=xr.DataArray(['hist-GHG1980', 'hist-GHG2004'], dims='simulation'))})
ds_coupledaer_JAS = xr.Dataset({"pr" : xr.concat([ds_aer1980_JAS.pr, ds_aer2004_JAS.pr], dim=xr.DataArray(['hist-aer1980', 'hist-aer2004'], dims='simulation'))})



In [6]:
# COMPUTING AREAS OF CELLS
area_grid_cell, dx, dy = area_grid(ds_forcedaer_JAS.lat, ds_forcedaer_JAS.lon)

ds_coupledaer_JAS['area_grid_cell'] = area_grid_cell
ds_coupledghg_JAS['area_grid_cell'] = area_grid_cell

ds_forcedghg_JAS['area_grid_cell'] = area_grid_cell
ds_forcedaer_JAS['area_grid_cell'] = area_grid_cell

ds_forced_JAS['area_grid_cell'] = area_grid_cell

In [7]:
# REMOVE LOCATIONS OVER SEA

sea_mask = sea_mask_creator(ds_ghg_JAS)

ds_forcedghg_JAS = ds_forcedghg_JAS.where(sea_mask == 0)
ds_forcedaer_JAS = ds_forcedaer_JAS.where(sea_mask == 0)

ds_coupledghg_JAS = ds_coupledghg_JAS.where(sea_mask == 0)
ds_coupledaer_JAS = ds_coupledaer_JAS.where(sea_mask == 0)

ds_forced_JAS = ds_forced_JAS.where(sea_mask == 0)



Creating sea mask: 100%|██████████| 1287/1287 [00:01<00:00, 978.82it/s]


In [10]:
# Number of rainy days

ds_forcedaer_JAS = n_I_rainy_days(ds_forcedaer_JAS, 'pr',1)
ds_forcedghg_JAS = n_I_rainy_days(ds_forcedghg_JAS, 'pr', 1)
ds_forced_JAS = n_I_rainy_days(ds_forced_JAS, 'pr', 1)

ds_forcedaer_JAS = WAM_moderate_heavy_extreme_difquant_forced(ds_forcedaer_JAS, 'pr', 1)
ds_forcedghg_JAS = WAM_moderate_heavy_extreme_difquant_forced(ds_forcedghg_JAS, 'pr', 1)



ds_coupledaer_JAS = n_I_rainy_days(ds_coupledaer_JAS, 'pr',1)
ds_coupledghg_JAS = n_I_rainy_days(ds_coupledghg_JAS, 'pr',1)

ds_coupledaer_JAS = WAM_moderate_heavy_extreme_difquant_forced(ds_coupledaer_JAS, 'pr',1)
ds_coupledghg_JAS = WAM_moderate_heavy_extreme_difquant_forced(ds_coupledghg_JAS, 'pr',1)

/home/pfernand/.conda/envs/xroms/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/home/pfernand/.conda/envs/xroms/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/home/pfernand/.conda/envs/xroms/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,
/home/pfernand/.conda/envs/xroms/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,


In [ ]:
# Intensity of rainy days

ds_coupledaer_JAS['I_rainy'] = ds_coupledaer_JAS.pr_rainy_cum_yr.mean(dim = ('year','member'), skipna = True) / ds_coupledaer_JAS.n_rainy.mean(dim = ('year', 'member'), skipna = True)
ds_coupledghg_JAS['I_rainy'] = ds_coupledghg_JAS.pr_rainy_cum_yr.mean(dim = ('year', 'member'), skipna = True) / ds_coupledghg_JAS.n_rainy.mean(dim = ('year', 'member'), skipna = True)

ds_forcedaer_JAS['I_rainy'] = ds_forcedaer_JAS.pr_rainy_cum_yr.mean(dim = ('year'), skipna = True) / ds_forcedaer_JAS.n_rainy.mean(dim = ('year'), skipna = True)
ds_forcedghg_JAS['I_rainy'] = ds_forcedghg_JAS.pr_rainy_cum_yr.mean(dim = ('year'), skipna = True) / ds_forcedghg_JAS.n_rainy.mean(dim = ('year'), skipna = True)


In [ ]:
# Compute differences between periods and statistical significance.

N_coupled = 92
N_forced = 90

ds_coupledaer_JAS['delta_p_20041980'] = 1 / N_coupled * (ds_coupledaer_JAS.pr_cum_yr.sel(simulation = 'hist-aer2004').mean(dim = ('year','member'), skipna = True) - ds_coupledaer_JAS.pr_cum_yr.sel(simulation = 'hist-aer1980').mean(dim = ('year','member'), skipna = True))
ds_coupledaer_JAS['pr_clim_20041980'] = 1 / N_coupled * (ds_coupledaer_JAS.pr_cum_yr.sel(simulation = 'hist-aer2004').mean(dim = ('year', 'member'), skipna = True) + ds_coupledaer_JAS.pr_cum_yr.sel(simulation = 'hist-aer1980').mean(dim = ('year', 'member'), skipna = True))/2



ds_coupledaer_JAS['delta_p_nr_20041980'] = 1 / N_coupled * (ds_coupledaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'hist-aer2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'hist-aer1980').mean(dim = ('year', 'member'), skipna = True))
ds_coupledaer_JAS['delta_n_20041980'] =  ds_coupledaer_JAS.n_rainy.sel(simulation = 'hist-aer2004').mean(dim = ('year','member'), skipna = True) - ds_coupledaer_JAS.n_rainy.sel(simulation = 'hist-aer1980').mean(dim = ('year','member'), skipna = True)


ds_coupledaer_JAS['delta_n_moderate_20041980'] =  ds_coupledaer_JAS.n_moderate.sel(simulation = 'hist-aer2004').mean(dim = ('year','member'), skipna = True) - ds_coupledaer_JAS.n_moderate.sel(simulation = 'hist-aer1980').mean(dim = ('year', 'member'), skipna = True)
ds_coupledaer_JAS['delta_n_heavy_20041980'] =  ds_coupledaer_JAS.n_heavy.sel(simulation = 'hist-aer2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledaer_JAS.n_heavy.sel(simulation = 'hist-aer1980').mean(dim = ('year', 'member'), skipna = True)


ds_coupledaer_JAS['delta_n_extreme_20041980'] =  ds_coupledaer_JAS.n_extreme.sel(simulation = 'hist-aer2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledaer_JAS.n_extreme.sel(simulation = 'hist-aer1980').mean(dim = ('year', 'member'), skipna = True)
ds_coupledaer_JAS['delta_I_20041980'] =  ds_coupledaer_JAS.I_rainy.sel(simulation = 'hist-aer2004') - ds_coupledaer_JAS.I_rainy.sel(simulation = 'hist-aer1980')


ds_coupledaer_JAS['n_clim_20041980'] =  (ds_coupledaer_JAS.n_rainy.sel(simulation = 'hist-aer2004').mean(dim = ('year', 'member'), skipna = True) + ds_coupledaer_JAS.n_rainy.sel(simulation = 'hist-aer1980').mean(dim = ('year', 'member'), skipna = True))/2
ds_coupledaer_JAS['I_clim_20041980'] =  (ds_coupledaer_JAS.I_rainy.sel(simulation = 'hist-aer2004') + ds_coupledaer_JAS.I_rainy.sel(simulation = 'hist-aer1980'))/2


ds_coupledaer_JAS['delta_p_n_20041980'] =  1 / N_coupled * ds_coupledaer_JAS.delta_n_20041980 * ds_coupledaer_JAS.I_clim_20041980
ds_coupledaer_JAS['delta_p_I_20041980'] =  1 / N_coupled * ds_coupledaer_JAS.delta_I_20041980 * ds_coupledaer_JAS.n_clim_20041980


############# For the GHG-RELATED simulations ###########################


ds_coupledghg_JAS['delta_p_20041980'] = 1 / N_coupled * (ds_coupledghg_JAS.pr_cum_yr.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledghg_JAS.pr_cum_yr.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True))
ds_coupledghg_JAS['pr_clim_20041980'] =  1 / N_coupled * (ds_coupledghg_JAS.pr_cum_yr.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) + ds_coupledghg_JAS.pr_cum_yr.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True))/2

ds_coupledghg_JAS['delta_p_nr_20041980'] = 1 / N_coupled * (ds_coupledghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True))
ds_coupledghg_JAS['delta_n_20041980'] =  ds_coupledghg_JAS.n_rainy.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledghg_JAS.n_rainy.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True)


ds_coupledghg_JAS['delta_n_moderate_20041980'] =  ds_coupledghg_JAS.n_moderate.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledghg_JAS.n_moderate.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True)
ds_coupledghg_JAS['delta_n_heavy_20041980'] =  ds_coupledghg_JAS.n_heavy.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledghg_JAS.n_heavy.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True)


ds_coupledghg_JAS['delta_n_extreme_20041980'] =  ds_coupledghg_JAS.n_extreme.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) - ds_coupledghg_JAS.n_extreme.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True)
ds_coupledghg_JAS['delta_I_20041980'] =  ds_coupledghg_JAS.I_rainy.sel(simulation = 'hist-GHG2004') - ds_coupledghg_JAS.I_rainy.sel(simulation = 'hist-GHG1980')


ds_coupledghg_JAS['n_clim_20041980'] =  (ds_coupledghg_JAS.n_rainy.sel(simulation = 'hist-GHG2004').mean(dim = ('year', 'member'), skipna = True) + ds_coupledghg_JAS.n_rainy.sel(simulation = 'hist-GHG1980').mean(dim = ('year', 'member'), skipna = True))/2
ds_coupledghg_JAS['I_clim_20041980'] =  (ds_coupledghg_JAS.I_rainy.sel(simulation = 'hist-GHG2004') + ds_coupledghg_JAS.I_rainy.sel(simulation = 'hist-GHG1980'))/2


ds_coupledghg_JAS['delta_p_n_20041980'] =  1 / N_coupled * ds_coupledghg_JAS.delta_n_20041980 * ds_coupledghg_JAS.I_clim_20041980
ds_coupledghg_JAS['delta_p_I_20041980'] =  1 / N_coupled * ds_coupledghg_JAS.delta_I_20041980 * ds_coupledghg_JAS.n_clim_20041980


############## For the forced simulations as well ##############################

ds_forcedghg_JAS['delta_p_20041980'] = 1 / N_forced * (ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_p_DA_20041980'] = 1 / N_forced * (ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_p_OM_20041980'] = 1 / N_forced * (ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))

ds_forcedghg_JAS['pr_clim_20041980'] = 1 / N_forced *(ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) + ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))/2
ds_forcedghg_JAS['pr_clim_DA_20041980'] =  1 / N_forced *(ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) + ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))/2
ds_forcedghg_JAS['pr_clim_OM_20041980'] =  1 / N_forced *(ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) + ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))/2





ds_forcedghg_JAS['delta_p_nr_20041980'] = 1 / N_forced * (ds_forcedghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_p_nr_DA_20041980'] = 1 / N_forced * (ds_forcedghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_p_nr_OM_20041980'] = 1 / N_forced * (ds_forcedghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))



ds_forcedghg_JAS['delta_n_20041980'] = (ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_DA_20041980'] =  (ds_forcedghg_JAS.n_rainy.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_OM_20041980'] =  (ds_forcedghg_JAS.n_rainy.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))



ds_forcedghg_JAS['delta_n_moderate_20041980'] = (ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_moderate_DA_20041980'] =  (ds_forcedghg_JAS.n_moderate.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_moderate_OM_20041980'] =  (ds_forcedghg_JAS.n_moderate.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))


ds_forcedghg_JAS['delta_n_heavy_20041980'] = (ds_forcedghg_JAS.n_heavy.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_heavy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_heavy_DA_20041980'] =  (ds_forcedghg_JAS.n_heavy.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_heavy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_heavy_OM_20041980'] =  (ds_forcedghg_JAS.n_heavy.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_heavy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))


ds_forcedghg_JAS['delta_n_extreme_20041980'] = (ds_forcedghg_JAS.n_extreme.sel(simulation = 'CTRLGHG2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_extreme.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_extreme_DA_20041980'] =  (ds_forcedghg_JAS.n_extreme.sel(simulation = 'GHGDA2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_extreme.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))
ds_forcedghg_JAS['delta_n_extreme_OM_20041980'] =  (ds_forcedghg_JAS.n_extreme.sel(simulation = 'GHGOM2004').mean(dim = ('year'), skipna = True) - ds_forcedghg_JAS.n_extreme.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))


ds_forcedghg_JAS['delta_I_20041980'] = (ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG2004') - ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980') )
ds_forcedghg_JAS['delta_I_DA_20041980'] =  (ds_forcedghg_JAS.I_rainy.sel(simulation = 'GHGDA2004')  - ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980') )
ds_forcedghg_JAS['delta_I_OM_20041980'] =  (ds_forcedghg_JAS.I_rainy.sel(simulation = 'GHGOM2004')  - ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980') )


ds_forcedghg_JAS['n_clim_20041980'] = ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True) #+ ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))/2
ds_forcedghg_JAS['n_clim_DA_20041980'] =  ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True) #+ ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))/2
ds_forcedghg_JAS['n_clim_OM_20041980'] =  ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True) #+ ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').mean(dim = ('year'), skipna = True))/2


ds_forcedghg_JAS['I_clim_20041980'] = ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980') #+ ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980'))/2
ds_forcedghg_JAS['I_clim_DA_20041980'] =  ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980') #+ ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980'))/2
ds_forcedghg_JAS['I_clim_OM_20041980'] =  ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980') #+ ds_forcedghg_JAS.I_rainy.sel(simulation = 'CTRLGHG1980'))/2


ds_forcedghg_JAS['delta_p_n_20041980'] = 1 / N_forced * ds_forcedghg_JAS.delta_n_20041980 * ds_forcedghg_JAS.I_clim_20041980
ds_forcedghg_JAS['delta_p_n_DA_20041980'] = 1 / N_forced * ds_forcedghg_JAS.delta_n_DA_20041980 * ds_forcedghg_JAS.I_clim_DA_20041980
ds_forcedghg_JAS['delta_p_n_OM_20041980'] = 1 / N_forced * ds_forcedghg_JAS.delta_n_OM_20041980 * ds_forcedghg_JAS.I_clim_OM_20041980

ds_forcedghg_JAS['delta_p_I_20041980'] = 1 / N_forced * ds_forcedghg_JAS.n_clim_20041980 * ds_forcedghg_JAS.delta_I_20041980
ds_forcedghg_JAS['delta_p_I_DA_20041980'] = 1 / N_forced * ds_forcedghg_JAS.n_clim_DA_20041980 * ds_forcedghg_JAS.delta_I_DA_20041980
ds_forcedghg_JAS['delta_p_I_OM_20041980'] = 1 / N_forced * ds_forcedghg_JAS.n_clim_OM_20041980 * ds_forcedghg_JAS.delta_I_OM_20041980

# For the aeorosols in the forced simulation


ds_forcedaer_JAS['delta_p_20041980'] = 1 / N_forced * (ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_p_DA_20041980'] = 1 / N_forced * (ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_p_OM_20041980'] = 1 / N_forced * (ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))


ds_forcedaer_JAS['pr_clim_20041980'] = 1 / N_forced *(ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) + ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))/2
ds_forcedaer_JAS['pr_clim_DA_20041980'] =  1 / N_forced *(ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) + ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))/2
ds_forcedaer_JAS['pr_clim_OM_20041980'] =  1 / N_forced *(ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) + ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))/2


ds_forcedaer_JAS['delta_p_nr_20041980'] = 1 / N_forced * (ds_forcedaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_p_nr_DA_20041980'] = 1 / N_forced * (ds_forcedaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_p_nr_OM_20041980'] = 1 / N_forced * (ds_forcedaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.pr_nonrainy_cum_yr.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))



ds_forcedaer_JAS['delta_n_20041980'] = (ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_DA_20041980'] =  (ds_forcedaer_JAS.n_rainy.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_OM_20041980'] =  (ds_forcedaer_JAS.n_rainy.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))



ds_forcedaer_JAS['delta_n_moderate_20041980'] = (ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_moderate_DA_20041980'] =  (ds_forcedaer_JAS.n_moderate.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_moderate_OM_20041980'] =  (ds_forcedaer_JAS.n_moderate.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))


ds_forcedaer_JAS['delta_n_heavy_20041980'] = (ds_forcedaer_JAS.n_heavy.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_heavy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_heavy_DA_20041980'] =  (ds_forcedaer_JAS.n_heavy.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_heavy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_heavy_OM_20041980'] =  (ds_forcedaer_JAS.n_heavy.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_heavy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))


ds_forcedaer_JAS['delta_n_extreme_20041980'] = (ds_forcedaer_JAS.n_extreme.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_extreme.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_extreme_DA_20041980'] =  (ds_forcedaer_JAS.n_extreme.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_extreme.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))
ds_forcedaer_JAS['delta_n_extreme_OM_20041980'] =  (ds_forcedaer_JAS.n_extreme.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) - ds_forcedaer_JAS.n_extreme.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))


ds_forcedaer_JAS['delta_I_20041980'] = (ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER2004') - ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER1980'))
ds_forcedaer_JAS['delta_I_DA_20041980'] =  (ds_forcedaer_JAS.I_rainy.sel(simulation = 'AERDA2004')  - ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER1980'))
ds_forcedaer_JAS['delta_I_OM_20041980'] =  (ds_forcedaer_JAS.I_rainy.sel(simulation = 'AEROM2004')  - ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER1980'))


ds_forcedaer_JAS['n_clim_20041980'] = (ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER2004').mean(dim = ('year'), skipna = True) + ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))/2
ds_forcedaer_JAS['n_clim_DA_20041980'] =  (ds_forcedaer_JAS.n_rainy.sel(simulation = 'AERDA2004').mean(dim = ('year'), skipna = True) + ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))/2
ds_forcedaer_JAS['n_clim_OM_20041980'] =  (ds_forcedaer_JAS.n_rainy.sel(simulation = 'AEROM2004').mean(dim = ('year'), skipna = True) + ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').mean(dim = ('year'), skipna = True))/2


ds_forcedaer_JAS['I_clim_20041980'] = (ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER2004') + ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER1980'))/2
ds_forcedaer_JAS['I_clim_DA_20041980'] =  (ds_forcedaer_JAS.I_rainy.sel(simulation = 'AERDA2004')+ ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER1980'))/2
ds_forcedaer_JAS['I_clim_OM_20041980'] =  (ds_forcedaer_JAS.I_rainy.sel(simulation = 'AEROM2004') + ds_forcedaer_JAS.I_rainy.sel(simulation = 'CTRLAER1980'))/2


ds_forcedaer_JAS['delta_p_n_20041980'] = 1 / N_forced * ds_forcedaer_JAS.delta_n_20041980 * ds_forcedaer_JAS.I_clim_20041980
ds_forcedaer_JAS['delta_p_n_DA_20041980'] = 1 / N_forced * ds_forcedaer_JAS.delta_n_DA_20041980 * ds_forcedaer_JAS.I_clim_DA_20041980
ds_forcedaer_JAS['delta_p_n_OM_20041980'] = 1 / N_forced * ds_forcedaer_JAS.delta_n_OM_20041980 * ds_forcedaer_JAS.I_clim_OM_20041980

ds_forcedaer_JAS['delta_p_I_20041980'] = 1 / N_forced * ds_forcedaer_JAS.n_clim_20041980 * ds_forcedaer_JAS.delta_I_20041980
ds_forcedaer_JAS['delta_p_I_DA_20041980'] = 1 / N_forced * ds_forcedaer_JAS.n_clim_DA_20041980 * ds_forcedaer_JAS.delta_I_DA_20041980
ds_forcedaer_JAS['delta_p_I_OM_20041980'] = 1 / N_forced * ds_forcedaer_JAS.n_clim_OM_20041980 * ds_forcedaer_JAS.delta_I_OM_20041980



In [ ]:
sea_mask = sea_mask_creator(ds_forcedghg_JAS)

ds_forcedghg_JAS = ds_forcedghg_JAS.where(sea_mask == 0)
ds_forcedaer_JAS = ds_forcedaer_JAS.where(sea_mask == 0)

ds_coupledghg_JAS = ds_coupledghg_JAS.where(sea_mask == 0)
ds_coupledaer_JAS = ds_coupledaer_JAS.where(sea_mask == 0)

In [ ]:
# COMPUTATION OF STATISTICAL SIGNIFICANCE 

############### PARA LA PRECIPITACIÓN TOTAL #############

t_stat_aer_pr_20041980, p_value_aer_pr_20041980 = stats.ttest_ind(ds_coupledaer_JAS.pr_rainy_cum_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer2004').values, ds_coupledaer_JAS.pr_rainy_cum_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer1980').values, axis = 0, equal_var=False)
t_stat_ctrlaer_pr_20041980, p_value_ctrlaer_pr_20041980 = stats.ttest_ind(ds_forcedaer_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLAER2004').values, ds_forcedaer_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)


t_stat_aerda_pr_20041980, p_value_aerda_pr_20041980 = stats.ttest_ind(ds_forcedaer_JAS.pr_rainy_cum_yr.sel(simulation = 'AERDA2004').values, ds_forcedaer_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerom_pr_20041980, p_value_aerom_pr_20041980 = stats.ttest_ind(ds_forcedaer_JAS.pr_rainy_cum_yr.sel(simulation = 'AEROM2004').values, ds_forcedaer_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)


############# NUMBER OF RAINY DAYS ###########


t_stat_aer_n_20041980, p_value_aer_n_20041980 = stats.ttest_ind(ds_coupledaer_JAS.n_rainy.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer2004').values, ds_coupledaer_JAS.n_rainy.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer1980').values, axis = 0, equal_var=False)

# Para la simulación forzada

t_stat_ctrlaer_n_20041980, p_value_ctrlaer_n_20041980 = stats.ttest_ind(ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER2004').values, ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerda_n_20041980, p_value_aerda_n_20041980 = stats.ttest_ind(ds_forcedaer_JAS.n_rainy.sel(simulation = 'AERDA2004').values, ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerom_n_20041980, p_value_aerom_n_20041980 = stats.ttest_ind(ds_forcedaer_JAS.n_rainy.sel(simulation = 'AEROM2004').values, ds_forcedaer_JAS.n_rainy.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)


############## INTENSITY OF RAINY DAYS #############

t_stat_aer_I_20041980, p_value_aer_I_20041980 = stats.ttest_ind(ds_coupledaer_JAS.I_rainy_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer2004').values, ds_coupledaer_JAS.I_rainy_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer1980').values, axis = 0, equal_var=False)

# Para la simulación forzada

t_stat_ctrlaer_I_20041980, p_value_ctrlaer_I_20041980 = stats.ttest_ind(ds_forcedaer_JAS.I_rainy_yr.sel(simulation = 'CTRLAER2004').values, ds_forcedaer_JAS.I_rainy_yr.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerda_I_20041980, p_value_aerda_I_20041980 = stats.ttest_ind(ds_forcedaer_JAS.I_rainy_yr.sel(simulation = 'AERDA2004').values, ds_forcedaer_JAS.I_rainy_yr.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerom_I_20041980, p_value_aerom_I_20041980 = stats.ttest_ind(ds_forcedaer_JAS.I_rainy_yr.sel(simulation = 'AEROM2004').values, ds_forcedaer_JAS.I_rainy_yr.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)





########## FOR MODERATE RAINY DAYS  ################


t_stat_aer_n_moderate_20041980, p_value_aer_n_moderate_20041980 = stats.ttest_ind(ds_coupledaer_JAS.n_moderate.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer2004').values, ds_coupledaer_JAS.n_moderate.mean(dim = 'member', skipna = True).sel(simulation = 'hist-aer1980').values, axis = 0, equal_var=False)
t_stat_ctrlaer_n_moderate_20041980, p_value_ctrlaer_n_moderate_20041980 = stats.ttest_ind(ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER2004').values, ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerda_n_moderate_20041980, p_value_aerda_n_moderate_20041980 = stats.ttest_ind(ds_forcedaer_JAS.n_moderate.sel(simulation = 'AERDA2004').values, ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)
t_stat_aerom_n_moderate_20041980, p_value_aerom_n_moderate_20041980 = stats.ttest_ind(ds_forcedaer_JAS.n_moderate.sel(simulation = 'AEROM2004').values, ds_forcedaer_JAS.n_moderate.sel(simulation = 'CTRLAER1980').values, axis = 0, equal_var=False)






################ FOR GHG SIMULATION ##################


t_stat_ghg_pr_20041980, p_value_ghg_pr_20041980 = stats.ttest_ind(ds_coupledghg_JAS.pr_rainy_cum_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG2004').values, ds_coupledghg_JAS.pr_rainy_cum_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG1980').values, axis = 0, equal_var=False)
t_stat_ctrlghg_pr_20041980, p_value_ctrlghg_pr_20041980 = stats.ttest_ind(ds_forcedghg_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLGHG2004').values, ds_forcedghg_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)


t_stat_ghgda_pr_20041980, p_value_ghgda_pr_20041980 = stats.ttest_ind(ds_forcedghg_JAS.pr_rainy_cum_yr.sel(simulation = 'GHGDA2004').values, ds_forcedghg_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgom_pr_20041980, p_value_ghgom_pr_20041980 = stats.ttest_ind(ds_forcedghg_JAS.pr_rainy_cum_yr.sel(simulation = 'GHGOM2004').values, ds_forcedghg_JAS.pr_rainy_cum_yr.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)


############# NUMBER OF RAINY DAYS ###########


t_stat_ghg_n_20041980, p_value_ghg_n_20041980 = stats.ttest_ind(ds_coupledghg_JAS.n_rainy.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG2004').values, ds_coupledghg_JAS.n_rainy.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG1980').values, axis = 0, equal_var=False)



t_stat_ctrlghg_n_20041980, p_value_ctrlghg_n_20041980 = stats.ttest_ind(ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG2004').values, ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgda_n_20041980, p_value_ghgda_n_20041980 = stats.ttest_ind(ds_forcedghg_JAS.n_rainy.sel(simulation = 'GHGDA2004').values, ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgom_n_20041980, p_value_ghgom_n_20041980 = stats.ttest_ind(ds_forcedghg_JAS.n_rainy.sel(simulation = 'GHGOM2004').values, ds_forcedghg_JAS.n_rainy.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)


############## FOR THE INTENSITY OF RAINY DAYS #############


t_stat_ghg_I_20041980, p_value_ghg_I_20041980 = stats.ttest_ind(ds_coupledghg_JAS.I_rainy_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG2004').values, ds_coupledghg_JAS.I_rainy_yr.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG1980').values, axis = 0, equal_var=False)

t_stat_ctrlghg_I_20041980, p_value_ctrlghg_I_20041980 = stats.ttest_ind(ds_forcedghg_JAS.I_rainy_yr.sel(simulation = 'CTRLGHG2004').values, ds_forcedghg_JAS.I_rainy_yr.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgda_I_20041980, p_value_ghgda_I_20041980 = stats.ttest_ind(ds_forcedghg_JAS.I_rainy_yr.sel(simulation = 'GHGDA2004').values, ds_forcedghg_JAS.I_rainy_yr.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgom_I_20041980, p_value_ghgom_I_20041980 = stats.ttest_ind(ds_forcedghg_JAS.I_rainy_yr.sel(simulation = 'GHGOM2004').values, ds_forcedghg_JAS.I_rainy_yr.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)





################ FOR MODERATE RAINY DAYS ##################


t_stat_ghg_n_moderate_20041980, p_value_ghg_n_moderate_20041980 = stats.ttest_ind(ds_coupledghg_JAS.n_moderate.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG2004').values, ds_coupledghg_JAS.n_moderate.mean(dim = 'member', skipna = True).sel(simulation = 'hist-GHG1980').values, axis = 0, equal_var=False)

t_stat_ctrlghg_n_moderate_20041980, p_value_ctrlghg_n_moderate_20041980 = stats.ttest_ind(ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG2004').values, ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgda_n_moderate_20041980, p_value_ghgda_n_moderate_20041980 = stats.ttest_ind(ds_forcedghg_JAS.n_moderate.sel(simulation = 'GHGDA2004').values, ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)
t_stat_ghgom_n_moderate_20041980, p_value_ghgom_n_moderate_20041980 = stats.ttest_ind(ds_forcedghg_JAS.n_moderate.sel(simulation = 'GHGOM2004').values, ds_forcedghg_JAS.n_moderate.sel(simulation = 'CTRLGHG1980').values, axis = 0, equal_var=False)




In [ ]:
# Plot the changes in precipitation for AER-RELATED SIMULATIONS

sahel_box_west = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':-10}
sahel_box_east = {'lat_min':8, 'lat_max':17, 'lon_min':-10, 'lon_max':10}
sahel_box_north = {'lat_min':13, 'lat_max':17, 'lon_min':-17, 'lon_max':10}
sahel_box_south = {'lat_min':8, 'lat_max':13, 'lon_min':-17, 'lon_max':10}
sahel_box = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':10}


levs = np.arange(-0.3, 0.35, 0.05)

extent = [-20,30,0,25]

fig = plt.figure(figsize=(60,40))

ft = 60
alpha = 0.05

plt.rcParams['hatch.linewidth'] = 3

ax1 = fig.add_subplot(1,3,1, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values, ds_forcedaer_JAS.delta_p_OM_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())
ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')


LON, LAT = np.meshgrid(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_aerom_n_20041980))| (p_value_aerom_n_20041980 > alpha) | (((ds_forcedaer_JAS.delta_p_n_OM_20041980) /  (ds_forcedaer_JAS.delta_p_OM_20041980) < 0.6)  |  ((ds_forcedaer_JAS.delta_p_n_OM_20041980) / (ds_forcedaer_JAS.delta_p_OM_20041980) <0))), LON)

hatch = ax1.contourf(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values, lon_masked, colors='none', hatches=['/'], transform=ccrs.PlateCarree())

LON, LAT = np.meshgrid(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_aerom_I_20041980))  | (p_value_aerom_I_20041980 > alpha)| (((ds_forcedaer_JAS.delta_p_I_OM_20041980) / (ds_forcedaer_JAS.delta_p_OM_20041980) < 0.6) | ((ds_forcedaer_JAS.delta_p_I_OM_20041980) / (ds_forcedaer_JAS.delta_p_OM_20041980) <0))), LON)

hatch = ax1.contourf(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values, lon_masked, colors='none', hatches=['-'], transform=ccrs.PlateCarree())




ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title(r'$\Delta$p', weight = 'bold', fontsize = ft) 





ax1 = fig.add_subplot(1,3,2, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values, ds_forcedaer_JAS.delta_p_n_OM_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())
ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')


LON, LAT = np.meshgrid(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values)
lon_masked = ma.masked_where((np.isnan(p_value_aerom_n_20041980)) | (p_value_aerom_n_20041980 > alpha), LON)
lat_masked = ma.masked_where((np.isnan(p_value_aerom_n_20041980)) | (p_value_aerom_n_20041980 > alpha), LAT)
hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)

ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title(r'$\frac{1}{N}$·$\Delta$n·$\overline{I}$', weight = 'bold', fontsize = ft) 



ax1 = fig.add_subplot(1,3,3, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values, ds_forcedaer_JAS.delta_p_I_OM_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())
ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')


LON, LAT = np.meshgrid(ds_ghg1960_JAS.lon.values, ds_ghg1960_JAS.lat.values)
lon_masked = ma.masked_where((np.isnan(p_value_aerom_I_20041980)) | (p_value_aerom_I_20041980 > alpha), LON)
lat_masked = ma.masked_where((np.isnan(p_value_aerom_I_20041980)) | (p_value_aerom_I_20041980 > alpha), LAT)
hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)

ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title(r'$\frac{1}{N}$·$\Delta$I·$\overline{n}$', weight = 'bold', fontsize = ft) 

fig.subplots_adjust(bottom=0.1)
cbar_ax = fig.add_axes([0.27, 0.3, 0.5, 0.02])
fig.colorbar(cax, cax=cbar_ax, orientation = 'horizontal',label=r'mm·d$^{-1}$')

fig.tight_layout()

#plt.savefig('/home/pfernand/Postdoc/Results/Forced_simulations/AER/JAS_precipitation_aerom_ctrl_100_yr_Sahel.jpg', bbox_inches = 'tight', dpi = 300)


In [ ]:
# Plot the figure for precipitation for the GHG-related simulations


sahel_box = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':10}
sahel_box_west = {'lat_min':12.5, 'lat_max':17, 'lon_min':-17, 'lon_max':-10}
sahel_box_east = {'lat_min':8, 'lat_max':17, 'lon_min':5, 'lon_max':10}


levs = np.arange(-0.3, 0.35, 0.05)

extent = [-20,30,0,25]

fig = plt.figure(figsize=(60,40))

font = {'family' : 'calibri',
       'weight': 'bold',
       'size': 60}
plt.rc('font', **font)
plt.rc('axes',edgecolor='k',linewidth=3)

ft = 60
alpha = 0.05

plt.rcParams['hatch.linewidth'] = 3

ax1 = fig.add_subplot(1,3,1, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, ds_forcedghg_JAS.delta_p_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())


ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box_east['lon_min'], sahel_box_east['lon_max'], sahel_box_east['lon_max'], sahel_box_east['lon_min'], sahel_box_east['lon_min']],  [sahel_box_east['lat_max'], sahel_box_east['lat_max'], sahel_box_east['lat_min'], sahel_box_east['lat_min'], sahel_box_east['lat_max']], linewidth = 7, color = 'purple')
ax1.plot([sahel_box_west['lon_min'], sahel_box_west['lon_max'], sahel_box_west['lon_max'], -15, sahel_box_west['lon_min'], sahel_box_west['lon_min']],  [sahel_box_west['lat_max'], sahel_box_west['lat_max'], sahel_box_west['lat_min'], sahel_box_west['lat_min'], 12.5, sahel_box_west['lat_max']], linewidth = 7, color = 'orange')


LON, LAT = np.meshgrid(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ctrlghg_n_20041980))| (p_value_ctrlghg_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_20041980) /  (ds_forcedghg_JAS.delta_p_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_20041980) / (ds_forcedghg_JAS.delta_p_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values, lon_masked, colors='none', hatches=['/'], transform=ccrs.PlateCarree())

LON, LAT = np.meshgrid(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ctrlghg_I_20041980))  | (p_value_ctrlghg_I_20041980 > alpha)| (((ds_forcedghg_JAS.delta_p_I_20041980) / (ds_forcedghg_JAS.delta_p_20041980) < 0.6) | ((ds_forcedghg_JAS.delta_p_I_20041980) / (ds_forcedghg_JAS.delta_p_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values, lon_masked, colors='none', hatches=['-'], transform=ccrs.PlateCarree())


# Pongo stippling fuera de las regiones significativas de antes pero con p significativa

lon_masked_n = ma.masked_where(((np.isnan(p_value_ctrlghg_n_20041980))| (p_value_ctrlghg_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_20041980) /  (ds_forcedghg_JAS.delta_p_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_20041980) / (ds_forcedghg_JAS.delta_p_20041980) <0))), LON)
lon_masked_I = ma.masked_where(((np.isnan(p_value_ctrlghg_I_20041980))| (p_value_ctrlghg_I_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_I_20041980) /  (ds_forcedghg_JAS.delta_p_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_I_20041980) / (ds_forcedghg_JAS.delta_p_20041980) <0))), LON)

lat_masked_n = ma.masked_where(((np.isnan(p_value_ctrlghg_n_20041980))| (p_value_ctrlghg_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_20041980) /  (ds_forcedghg_JAS.delta_p_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_20041980) / (ds_forcedghg_JAS.delta_p_20041980) <0))), LAT)
lat_masked_I = ma.masked_where(((np.isnan(p_value_ctrlghg_I_20041980))| (p_value_ctrlghg_I_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_I_20041980) /  (ds_forcedghg_JAS.delta_p_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_I_20041980) / (ds_forcedghg_JAS.delta_p_20041980) <0))), LAT)


inverted_lon_n = ma.masked_array(lon_masked_n.data, mask=~lon_masked_n.mask)
inverted_lat_n = ma.masked_array(lat_masked_n.data, mask=~lat_masked_n.mask)

inverted_lon_I = ma.masked_array(lon_masked_I.data, mask=~lon_masked_I.mask)
inverted_lat_I = ma.masked_array(lat_masked_I.data, mask=~lat_masked_I.mask)



lon_masked = ma.masked_where((np.isnan(p_value_ctrlghg_pr_20041980)) | (p_value_ctrlghg_pr_20041980 > alpha), (inverted_lon_n + inverted_lon_I + LON)/3)
lat_masked = ma.masked_where((np.isnan(p_value_ctrlghg_pr_20041980)) | (p_value_ctrlghg_pr_20041980 > alpha), (inverted_lat_n + inverted_lat_I + LAT)/3)

hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)


ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title(r'Total Response', weight = 'bold', fontsize = ft+10) 




ax1 = fig.add_subplot(1,3,2, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, ds_forcedghg_JAS.delta_p_DA_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())
ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box_east['lon_min'], sahel_box_east['lon_max'], sahel_box_east['lon_max'], sahel_box_east['lon_min'], sahel_box_east['lon_min']],  [sahel_box_east['lat_max'], sahel_box_east['lat_max'], sahel_box_east['lat_min'], sahel_box_east['lat_min'], sahel_box_east['lat_max']], linewidth = 7, color = 'purple')
ax1.plot([sahel_box_west['lon_min'], sahel_box_west['lon_max'], sahel_box_west['lon_max'], -15, sahel_box_west['lon_min'], sahel_box_west['lon_min']],  [sahel_box_west['lat_max'], sahel_box_west['lat_max'], sahel_box_west['lat_min'], sahel_box_west['lat_min'], 12.5, sahel_box_west['lat_max']], linewidth = 7, color = 'orange')


LON, LAT = np.meshgrid(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ghgda_n_20041980))| (p_value_ghgda_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_DA_20041980) /  (ds_forcedghg_JAS.delta_p_DA_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values, lon_masked, colors='none', hatches=['/'], transform=ccrs.PlateCarree())

LON, LAT = np.meshgrid(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ghgda_I_20041980))  | (p_value_ghgda_I_20041980 > alpha)| (((ds_forcedghg_JAS.delta_p_I_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) < 0.6) | ((ds_forcedghg_JAS.delta_p_I_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, lon_masked, colors='none', hatches=['-'], transform=ccrs.PlateCarree())


# Pongo stippling fuera de las regiones significativas de antes pero con p significativa

lon_masked_n = ma.masked_where(((np.isnan(p_value_ghgda_n_20041980))| (p_value_ghgda_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_DA_20041980) /  (ds_forcedghg_JAS.delta_p_DA_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) <0))), LON)
lon_masked_I = ma.masked_where(((np.isnan(p_value_ghgda_I_20041980))| (p_value_ghgda_I_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_I_DA_20041980) /  (ds_forcedghg_JAS.delta_p_DA_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_I_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) <0))), LON)

lat_masked_n = ma.masked_where(((np.isnan(p_value_ghgda_n_20041980))| (p_value_ghgda_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_DA_20041980) /  (ds_forcedghg_JAS.delta_p_DA_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) <0))), LAT)
lat_masked_I = ma.masked_where(((np.isnan(p_value_ghgda_I_20041980))| (p_value_ghgda_I_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_I_DA_20041980) /  (ds_forcedghg_JAS.delta_p_DA_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_I_DA_20041980) / (ds_forcedghg_JAS.delta_p_DA_20041980) <0))), LAT)


inverted_lon_n = ma.masked_array(lon_masked_n.data, mask=~lon_masked_n.mask)
inverted_lat_n = ma.masked_array(lat_masked_n.data, mask=~lat_masked_n.mask)

inverted_lon_I = ma.masked_array(lon_masked_I.data, mask=~lon_masked_I.mask)
inverted_lat_I = ma.masked_array(lat_masked_I.data, mask=~lat_masked_I.mask)



lon_masked = ma.masked_where((np.isnan(p_value_ghgda_pr_20041980)) | (p_value_ghgda_pr_20041980 > alpha), (inverted_lon_n + inverted_lon_I + LON)/3)
lat_masked = ma.masked_where((np.isnan(p_value_ghgda_pr_20041980)) | (p_value_ghgda_pr_20041980 > alpha), (inverted_lat_n + inverted_lat_I + LAT)/3)

hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)





ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title(r'DA', weight = 'bold', fontsize = ft+10) 



ax1 = fig.add_subplot(1,3,3, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values, ds_forcedghg_JAS.delta_p_OM_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())
ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box_east['lon_min'], sahel_box_east['lon_max'], sahel_box_east['lon_max'], sahel_box_east['lon_min'], sahel_box_east['lon_min']],  [sahel_box_east['lat_max'], sahel_box_east['lat_max'], sahel_box_east['lat_min'], sahel_box_east['lat_min'], sahel_box_east['lat_max']], linewidth = 7, color = 'purple')
ax1.plot([sahel_box_west['lon_min'], sahel_box_west['lon_max'], sahel_box_west['lon_max'], -15, sahel_box_west['lon_min'], sahel_box_west['lon_min']],  [sahel_box_west['lat_max'], sahel_box_west['lat_max'], sahel_box_west['lat_min'], sahel_box_west['lat_min'], 12.5, sahel_box_west['lat_max']], linewidth = 7, color = 'orange')


LON, LAT = np.meshgrid(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ghgom_n_20041980))| (p_value_ghgom_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_OM_20041980) /  (ds_forcedghg_JAS.delta_p_OM_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, lon_masked, colors='none', hatches=['/'], transform=ccrs.PlateCarree())

LON, LAT = np.meshgrid(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ghgom_I_20041980))  | (p_value_ghgom_I_20041980 > alpha)| (((ds_forcedghg_JAS.delta_p_I_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) < 0.6) | ((ds_forcedghg_JAS.delta_p_I_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values, lon_masked, colors='none', hatches=['-'], transform=ccrs.PlateCarree())


# Pongo stippling fuera de las regiones significativas de antes pero con p significativa

lon_masked_n = ma.masked_where(((np.isnan(p_value_ghgom_n_20041980))| (p_value_ghgom_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_OM_20041980) /  (ds_forcedghg_JAS.delta_p_OM_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) <0))), LON)
lon_masked_I = ma.masked_where(((np.isnan(p_value_ghgom_I_20041980))| (p_value_ghgom_I_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_I_OM_20041980) /  (ds_forcedghg_JAS.delta_p_OM_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_I_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) <0))), LON)

lat_masked_n = ma.masked_where(((np.isnan(p_value_ghgom_n_20041980))| (p_value_ghgom_n_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_n_OM_20041980) /  (ds_forcedghg_JAS.delta_p_OM_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_n_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) <0))), LAT)
lat_masked_I = ma.masked_where(((np.isnan(p_value_ghgom_I_20041980))| (p_value_ghgom_I_20041980 > alpha) | (((ds_forcedghg_JAS.delta_p_I_OM_20041980) /  (ds_forcedghg_JAS.delta_p_OM_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_I_OM_20041980) / (ds_forcedghg_JAS.delta_p_OM_20041980) <0))), LAT)


inverted_lon_n = ma.masked_array(lon_masked_n.data, mask=~lon_masked_n.mask)
inverted_lat_n = ma.masked_array(lat_masked_n.data, mask=~lat_masked_n.mask)

inverted_lon_I = ma.masked_array(lon_masked_I.data, mask=~lon_masked_I.mask)
inverted_lat_I = ma.masked_array(lat_masked_I.data, mask=~lat_masked_I.mask)



lon_masked = ma.masked_where((np.isnan(p_value_ghgom_pr_20041980)) | (p_value_ghgom_pr_20041980 > alpha), (inverted_lon_n + inverted_lon_I + LON)/3)
lat_masked = ma.masked_where((np.isnan(p_value_ghgom_pr_20041980)) | (p_value_ghgom_pr_20041980 > alpha), (inverted_lat_n + inverted_lat_I + LAT)/3)

hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)




ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title(r'OM', weight = 'bold', fontsize = ft+10) 


fig.subplots_adjust(bottom=0.1)
cbar_ax = fig.add_axes([0.27, 0.3, 0.5, 0.02])
fig.colorbar(cax, cax=cbar_ax, orientation = 'horizontal',label=r'$\Delta$p [mm·d$^{-1}$]')
cbar_ax.tick_params(labelsize=ft)

fig.tight_layout()


#plt.savefig('/home/pfernand/Postdoc/Results/Forced_simulations/GHG/JAS_delta_p_GHG_ctrl_100_yr_Sahel.jpg', bbox_inches = 'tight', dpi = 150)


### Linear additivity between forcings

In [ ]:
# Linear additivity between forcings

ds_forced_JAS['pr_cum_yr_GHGAA_1980'] = ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER1980') + ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG1980')
ds_forced_JAS['pr_cum_yr_GHGAA_2004'] = ds_forcedghg_JAS.pr_cum_yr.sel(simulation = 'CTRLGHG2004') + ds_forcedaer_JAS.pr_cum_yr.sel(simulation = 'CTRLAER2004')

ds_forced_JAS['delta_p_20041980'] = 1 / N_forced * (ds_forced_JAS.pr_cum_yr.sel(simulation = 'CTRL2004').mean(dim = 'year', skipna = True) - ds_forced_JAS.pr_cum_yr.sel(simulation = 'CTRL1980').mean(dim = 'year', skipna = True))
ds_forced_JAS['delta_p_GHGAA_20041980'] = 1 / N_forced * (ds_forced_JAS.pr_cum_yr_GHGAA_2004.mean(dim = 'year', skipna = True) - ds_forced_JAS.pr_cum_yr_GHGAA_1980.mean(dim = 'year', skipna = True))


# Calculo el término no lineal

ds_forced_JAS['delta_p_nl_20041980'] = ds_forced_JAS['delta_p_20041980'] - ds_forced_JAS['delta_p_GHGAA_20041980']

In [ ]:
# Calculo las significaciones estadísticas

alpha = 0.05
t_stat_ctrl_pr_20041980, p_value_ctrl_pr_20041980 = stats.ttest_ind(ds_forced_JAS.pr_cum_yr.sel(simulation = 'CTRL2004').values , ds_forced_JAS.pr_cum_yr.sel(simulation = 'CTRL1980').values, axis = 0, equal_var=False)
t_stat_ctrl_GHGAA_pr_20041980, p_value_ctrl_GHGAA_pr_20041980 = stats.ttest_ind(ds_forced_JAS.pr_cum_yr_GHGAA_2004.values , ds_forced_JAS.pr_cum_yr_GHGAA_1980.values, axis = 0, equal_var=False)


In [ ]:
# También pinto una figura que contenga únicamente los cambios de la precipitación


sahel_box = {'lat_min':8, 'lat_max':17, 'lon_min':-17, 'lon_max':10}
sahel_box_west = {'lat_min':12.5, 'lat_max':17, 'lon_min':-17, 'lon_max':-10}
sahel_box_east = {'lat_min':8, 'lat_max':17, 'lon_min':5, 'lon_max':10}


levs = np.arange(-0.6, 0.7, 0.1)

extent = [-20,30,0,25]

fig = plt.figure(figsize=(60,40))

font = {'family' : 'calibri',
       'weight': 'bold',
       'size': 60}
plt.rc('font', **font)
plt.rc('axes',edgecolor='k',linewidth=3)

ft = 60
alpha = 0.05

plt.rcParams['hatch.linewidth'] = 3

ax1 = fig.add_subplot(1,2,1, projection = ccrs.PlateCarree(0))

pr_masked = lon_masked = ma.masked_where((np.isnan(p_value_ctrl_pr_20041980)), ds_forced_JAS.delta_p_20041980.values)
cax=ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, pr_masked, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())


ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box_east['lon_min'], sahel_box_east['lon_max'], sahel_box_east['lon_max'], sahel_box_east['lon_min'], sahel_box_east['lon_min']],  [sahel_box_east['lat_max'], sahel_box_east['lat_max'], sahel_box_east['lat_min'], sahel_box_east['lat_min'], sahel_box_east['lat_max']], linewidth = 7, color = 'purple')
ax1.plot([sahel_box_west['lon_min'], sahel_box_west['lon_max'], sahel_box_west['lon_max'], -15, sahel_box_west['lon_min'], sahel_box_west['lon_min']],  [sahel_box_west['lat_max'], sahel_box_west['lat_max'], sahel_box_west['lat_min'], sahel_box_west['lat_min'], 12.5, sahel_box_west['lat_max']], linewidth = 7, color = 'orange')



LON, LAT = np.meshgrid(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values)
lon_masked = ma.masked_where(((np.isnan(p_value_ctrl_pr_20041980<0.05)) | ((np.abs(ds_forced_JAS.delta_p_nl_20041980) /  np.abs(ds_forced_JAS.delta_p_20041980) > 0.4)  |  ((ds_forced_JAS.delta_p_nl_20041980) / (ds_forced_JAS.delta_p_20041980) <0))), LON)


hatch = ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, lon_masked, colors='none', hatches=['|'], transform=ccrs.PlateCarree())

LON, LAT = np.meshgrid(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values)
lon_masked = ma.masked_where((((p_value_ctrl_pr_20041980<0.05)) | ((np.abs(ds_forced_JAS.delta_p_nl_20041980) /  np.abs(ds_forced_JAS.delta_p_20041980) < 0.6)  |  ((ds_forced_JAS.delta_p_nl_20041980) / (ds_forced_JAS.delta_p_20041980) <0))), LON)
lat_masked = ma.masked_where((((p_value_ctrl_pr_20041980<0.05)) | ((np.abs(ds_forced_JAS.delta_p_nl_20041980) /  np.abs(ds_forced_JAS.delta_p_20041980) < 0.6)  |  ((ds_forced_JAS.delta_p_nl_20041980) / (ds_forced_JAS.delta_p_20041980) <0))), LAT)



hatch = ax1.plot(np.transpose(lon_masked),np.transpose(lat_masked),'.',color='black',markersize=20)






ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title('Linear + Nonlinear Responses', weight = 'bold', fontsize = ft+20) 




ax1 = fig.add_subplot(1,2,2, projection = ccrs.PlateCarree(0))

cax=ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, ds_forced_JAS.delta_p_GHGAA_20041980.values, levels = levs, extend = 'both', cmap="BrBG", transform=ccrs.PlateCarree())

ax1.plot([sahel_box['lon_min'], sahel_box['lon_max'], sahel_box['lon_max'], -12.5, sahel_box['lon_min'], sahel_box['lon_min']],  [sahel_box['lat_max'], sahel_box['lat_max'], sahel_box['lat_min'], sahel_box['lat_min'], 12, sahel_box['lat_max']], linewidth = 7, color = 'blue')
ax1.plot([sahel_box_east['lon_min'], sahel_box_east['lon_max'], sahel_box_east['lon_max'], sahel_box_east['lon_min'], sahel_box_east['lon_min']],  [sahel_box_east['lat_max'], sahel_box_east['lat_max'], sahel_box_east['lat_min'], sahel_box_east['lat_min'], sahel_box_east['lat_max']], linewidth = 7, color = 'purple')
ax1.plot([sahel_box_west['lon_min'], sahel_box_west['lon_max'], sahel_box_west['lon_max'], -15, sahel_box_west['lon_min'], sahel_box_west['lon_min']],  [sahel_box_west['lat_max'], sahel_box_west['lat_max'], sahel_box_west['lat_min'], sahel_box_west['lat_min'], 12.5, sahel_box_west['lat_max']], linewidth = 7, color = 'orange')





LON, LAT = np.meshgrid(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values)


lon_masked = ma.masked_where(((p_value_ctrl_GHGAA_pr_20041980<0.05)) | ((np.abs(ds_forcedaer_JAS.delta_p_20041980) /  np.abs(ds_forced_JAS.delta_p_GHGAA_20041980) < 0.6)  |  ((ds_forcedaer_JAS.delta_p_20041980) / (ds_forced_JAS.delta_p_GHGAA_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values, lon_masked, colors='none', hatches=['/'], transform=ccrs.PlateCarree())

LON, LAT = np.meshgrid(ds_forcedaer_JAS.lon.values, ds_forcedaer_JAS.lat.values)

lon_masked = ma.masked_where((((p_value_ctrl_GHGAA_pr_20041980<0.05)) | ((np.abs(ds_forcedghg_JAS.delta_p_20041980) /  np.abs(ds_forced_JAS.delta_p_GHGAA_20041980) < 0.6)  |  ((ds_forcedghg_JAS.delta_p_20041980) / (ds_forced_JAS.delta_p_GHGAA_20041980) <0))), LON)

hatch = ax1.contourf(ds_forcedghg_JAS.lon.values, ds_forcedghg_JAS.lat.values, lon_masked, colors='none', hatches=['-'], transform=ccrs.PlateCarree())







ax1.coastlines(zorder=11,linewidth = 2)
ax1.set_extent(extent)

gl = ax1.gridlines(draw_labels=True, linestyle='--', color='gray', linewidth=2)
gl.xlocator = MultipleLocator(5)
gl.ylocator = MultipleLocator(5)
gl.top_labels = False
gl.right_labels = False
gl.bottom_labels = True
gl.left_labels = True
plt.rc('xtick', labelsize=40)
plt.rc('ytick', labelsize=40)
ax1.set_title('Linear Response', weight = 'bold', fontsize = ft+20) 


fig.subplots_adjust(bottom=0.1)
cbar_ax = fig.add_axes([0.27, 0.25, 0.5, 0.02])
fig.colorbar(cax, cax=cbar_ax, orientation = 'horizontal',label=r'$\Delta$p [mm·d$^{-1}$]')
cbar_ax.tick_params(labelsize=ft)

fig.tight_layout()


plt.savefig('/home/pfernand/Postdoc/Results/Forced_simulations/Linearity_between_forcings_new.jpg', bbox_inches = 'tight', dpi = 150)
